# Updated Targeted Metabolomics Workflow Demo

## Single RT Alignment with Individual Analysis Notebooks

This notebook demonstrates the updated targeted metabolomics workflow structure that:

1. **Single RT Alignment**: Uses one RT alignment atlas to create a single correction model
2. **Analysis-Specific Processing**: Applies the model to multiple analysis atlases (QC, ISTD, EMA)
3. **Individual Notebooks**: Creates separate notebooks for each analysis type/polarity combination
4. **Streamlined Curation**: Enables focused manual curation per analysis

### Key Features:
- **One Model, Many Atlases**: Single RT correction model applied to all analysis atlases
- **Subset Support**: Can specify analysis subset like `[('ISTD', 'NEG'), ('ISTD', 'POS')]`
- **Individual Curation**: Separate notebooks for each analysis type/polarity
- **Efficient Caching**: Cached results for rapid re-runs and testing

**Generated:** 2024-01-15 10:30:00

In [1]:
# Import required libraries
import sys
import os
from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# Add metatlas2 to Python path
sys.path.append('/Users/BKieft/Metabolomics/metatlas2/metatlas2')

# Import metatlas2 modules
import workflow_objects as wfo
import load_tools as ldt
import logging_config as lcf
import database_interact as dbi

# Setup logging
logger = lcf.get_logger('workflow_demo')

print("✅ All imports successful!")
print(f"Python path includes: {'/Users/BKieft/Metabolomics/metatlas2/metatlas2' in sys.path}")
print(f"Current working directory: {os.getcwd()}")

✅ All imports successful!
Python path includes: True
Current working directory: /Users/BKieft/Metabolomics/metatlas2


## Section 1: Load Configuration and Setup Workflow

Load the updated YAML configuration, initialize the TargetedMetabolomicsWorkflow object, and validate that the RT alignment atlas and all analysis atlases exist in the databases.

In [3]:
# Load configuration with updated structure
config_path = '/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml'
config = ldt.load_metatlas2_config(config_path)

# Display the new workflow configuration structure
print("📋 WORKFLOW CONFIGURATION STRUCTURE")
print("=" * 50)

workflow_config = config.get('WORKFLOWS', {}).get('HILICZ', {})

# Show RT alignment configuration (single atlas)
rt_align_config = workflow_config.get('RT_ALIGN', {})
rt_align_atlas = rt_align_config.get('ATLAS', {}).get('uid')
print(f"🎯 RT ALIGNMENT ATLAS: {rt_align_atlas}")
print(f"   Parameters: {rt_align_config.get('PARAMS', {})}")

# Show analysis atlases (multiple, organized by type and polarity)
analyses_config = workflow_config.get('ANALYSES', {})
print(f"\n📊 ANALYSIS ATLASES:")
for atlas_type, methods in analyses_config.items():
    print(f"  {atlas_type}:")
    for polarity, atlas_config in methods.items():
        atlas_uid = atlas_config.get('ATLAS', {}).get('uid')
        print(f"    {polarity}: {atlas_uid}")

print(f"\n✅ Configuration loaded successfully from: {config_path}")

2025-09-12 14:41:49 | metatlas2.load_tools | INFO | Loaded metatlas2 configuration from /Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml


📋 WORKFLOW CONFIGURATION STRUCTURE
🎯 RT ALIGNMENT ATLAS: atl-raw-qc-18c8d7b192964aedad11a0a8cf849fe4
   Parameters: {'ppm_error': 20.0, 'extra_time': 1.0, 'apply_model_to_min_max': True, 'polynomial_degree': 2, 'min_observations_per_compound': 1, 'min_compounds_for_modeling': 2, 'r2_threshold': 0.7, 'exclude_inchikeys': ['OVRNDRQMDRJTHS-ZEUBEQSHSA-N']}

📊 ANALYSIS ATLASES:
  QC:
    POS: atl-raw-qc-18c8d7b192964aedad11a0a8cf849fe4
    NEG: None
  ISTD:
    POS: atl-raw-istd-bd5844f5fb7746d0b5d1910d8a920a07
    NEG: None
  EMA:
    POS: atl-raw-ema-16130431dae94824adfc55fa19ebedd1
    NEG: None

✅ Configuration loaded successfully from: /Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml


In [ ]:
# Initialize the updated workflow object
project_name = "Demo_Project_2024"
project_directory = f"/Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/{project_name}"
project_db_path = f"{project_directory}/{project_name}.duckdb"
project_lcmsruns_path = f"{project_directory}/lcms_runs"

# Create project directory if it doesn't exist
Path(project_directory).mkdir(parents=True, exist_ok=True)

# Initialize the workflow with updated structure
workflow = wfo.TargetedMetabolomicsWorkflow(
    config=config,
    project_db_path=project_db_path,
    project_directory=project_directory,
    project_name=project_name,
    project_lcmsruns_path=project_lcmsruns_path
)

print("🚀 WORKFLOW INITIALIZATION")
print("=" * 50)
print(f"Project: {project_name}")
print(f"Directory: {project_directory}")
print(f"Database: {project_db_path}")
print(f"Current stage: {workflow.current_stage.value}")

# Display parsed atlas data
print(f"\n📚 PARSED ATLAS DATA:")
print(f"RT Alignment Atlas: {workflow.atlas_data.get('rt_align_atlas')}")
print(f"Analysis Atlases: {len(workflow.atlas_data.get('analysis_atlases', []))}")

for atlas in workflow.atlas_data.get('analysis_atlases', []):
    print(f"  - {atlas['atlas_type'].upper()} {atlas['polarity'].upper()}: {atlas['atlas_uid']}")

print("\n✅ Workflow initialized successfully!")

## Section 2: Run Single RT Alignment Model

Extract QC matches from the single RT alignment atlas, build a single polynomial RT correction model using QC files, and save the model to the database for reuse across all analyses.

In [ ]:
# Run the complete workflow up to RT correction stage
print("⚡ RUNNING SINGLE RT ALIGNMENT")
print("=" * 50)

try:
    # This will run stages 1 and 2: Project setup and RT correction
    workflow.run_complete_workflow(
        stop_at_stage=wfo.WorkflowStage.RT_CORRECTION
    )
    
    # Display RT correction results
    rt_summary = workflow.results.get_rt_correction_summary()
    print(f"✅ RT correction completed successfully!")
    print(f"   Models created: {rt_summary['rt_models_count']}")
    print(f"   Methods corrected: {rt_summary['methods_corrected']}")
    print(f"   Corrected atlases: {rt_summary['corrected_atlases']}")
    
    # Show the single RT model details
    for method, model in workflow.results.rt_models.items():
        print(f"\n📊 RT MODEL for {method}:")
        print(f"   R²: {model.get('r2', 'N/A'):.4f}")
        print(f"   RMSE: {model.get('rmse', 'N/A'):.4f}")
        print(f"   Polynomial degree: {model.get('degree', 'N/A')}")
        print(f"   Compounds used: {model.get('n_compounds', 'N/A')}")
        print(f"   Data points: {model.get('n_data_points', 'N/A')}")
    
except Exception as e:
    print(f"❌ Error during RT correction: {e}")
    print("This is expected in demo mode without actual data files.")

## Section 3: Apply RT Correction to Analysis Atlases

Apply the single RT correction model to all analysis atlases (QC, ISTD, EMA for both positive and negative polarities), create RT-corrected versions of each atlas, and save them to the project database.

In [ ]:
# Demonstrate RT correction application to all analysis atlases
print("🔄 RT CORRECTION APPLICATION")
print("=" * 50)

if workflow.results.has_rt_correction_results():
    print("✅ RT correction model available")
    
    # Show how the single model is applied to multiple analysis atlases
    print(f"\n📋 APPLYING SINGLE MODEL TO ANALYSIS ATLASES:")
    
    corrected_atlases = workflow.results.corrected_atlases
    for atlas_type, chrom_methods in corrected_atlases.items():
        print(f"\n{atlas_type.upper()}:")
        for chrom, polarities in chrom_methods.items():
            for polarity, corrected_uid in polarities.items():
                original_atlas = None
                # Find original atlas
                for analysis_atlas in workflow.atlas_data.get('analysis_atlases', []):
                    if (analysis_atlas['atlas_type'] == atlas_type and 
                        analysis_atlas['polarity'] == polarity):
                        original_atlas = analysis_atlas['atlas_uid']
                        break
                
                print(f"  {polarity.upper()}:")
                print(f"    Original: {original_atlas}")
                print(f"    RT-corrected: {corrected_uid}")
    
    print(f"\n🎯 KEY INSIGHT: One RT model applied to {len([atlas for atlases in corrected_atlases.values() for chrom in atlases.values() for atlas in chrom.values()])} analysis atlases")
    
else:
    print("ℹ️  RT correction not yet completed")
    print("   In actual workflow, this would show:")
    print("   - Single polynomial model parameters")
    print("   - Application to each analysis atlas")
    print("   - RT-corrected atlas UIDs saved to database")
    print("   - Statistics on RT shifts applied")

## Section 4: Run Putative Identification Analysis

Run targeted analysis workflow on each RT-corrected analysis atlas, extract putative identifications with EIC and MS/MS data, and store results organized by atlas type and polarity.

In [ ]:
# Run putative identification analysis with subset support
print("🔍 PUTATIVE IDENTIFICATION ANALYSIS")
print("=" * 50)

# Example 1: Run ALL analyses (default behavior)
print("📊 EXAMPLE 1: All Analyses (Default)")
try:
    # This would run putative identification on all analysis atlases
    workflow.run_complete_workflow(
        stop_at_stage=wfo.WorkflowStage.PUTATIVE_IDENTIFICATION,
        analysis_subset=None  # None = all analyses
    )
    
    # Display results
    if workflow.results.has_putative_identification_results():
        stats = workflow.results.get_putative_identification_summary()
        print(f"✅ Putative identification completed!")
        print(f"   Total identifications: {stats['total_putative_ids']}")
        print(f"   By atlas type: {stats['by_atlas_type']}")
        print(f"   With MS/MS data: {stats['with_ms2_data']}")
        print(f"   With reference hits: {stats['with_reference_hits']}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("   Expected in demo without data files")

# Example 2: Run SUBSET of analyses
print(f"\n📊 EXAMPLE 2: Subset Analyses")
print("   Demonstrating analysis_subset parameter...")

# Define subset - only ISTD positive and negative
analysis_subset = [('istd', 'pos'), ('istd', 'neg')]
print(f"   Subset: {analysis_subset}")

try:
    # This would run only the specified subset
    workflow.run_complete_workflow(
        stop_at_stage=wfo.WorkflowStage.PUTATIVE_IDENTIFICATION,
        analysis_subset=analysis_subset
    )
    
    print(f"✅ Subset analysis would complete with {len(analysis_subset)} analyses")
    
except Exception as e:
    print(f"   Note: {e}")

print(f"\n🎯 KEY FEATURE: analysis_subset parameter allows focused analysis")
print(f"   Useful for debugging, testing, or focused studies")

## Section 5: Generate Individual Analysis Notebooks

Create separate Jupyter notebooks for each analysis type/polarity combination (e.g., QC_POS, ISTD_NEG, EMA_POS), with each notebook containing manual curation GUI and final reporting specific to that analysis.

In [ ]:
# Generate individual notebooks for each analysis type/polarity
print("📓 GENERATING INDIVIDUAL ANALYSIS NOTEBOOKS")
print("=" * 50)

# This demonstrates the key new feature: individual notebooks per analysis
try:
    # Generate individual notebooks for each analysis
    created_notebooks = workflow.run_complete_workflow(
        stop_at_stage=wfo.WorkflowStage.PUTATIVE_IDENTIFICATION,
        create_individual_notebooks=True  # This creates separate notebooks
    )
    
    if created_notebooks:
        print(f"✅ Created {len(created_notebooks)} individual analysis notebooks:")
        for notebook_path in created_notebooks:
            notebook_name = Path(notebook_path).name
            print(f"   📝 {notebook_name}")
    
except Exception as e:
    print(f"ℹ️  Demo mode - notebooks would be created in actual workflow")
    
    # Show what WOULD be created
    print(f"   Individual notebooks that would be generated:")
    
    # Simulate notebook creation based on analysis atlases
    for analysis_atlas in workflow.atlas_data.get('analysis_atlases', []):
        atlas_type = analysis_atlas['atlas_type']
        polarity = analysis_atlas['polarity']
        notebook_name = f"curation_{atlas_type}_{polarity}.ipynb"
        print(f"   📝 {notebook_name}")
        print(f"      - Interactive curation GUI for {atlas_type.upper()} {polarity.upper()}")
        print(f"      - Auto-save progress during curation")
        print(f"      - Analysis-specific summary export")

# Show the structure of individual notebooks
print(f"\n📋 INDIVIDUAL NOTEBOOK STRUCTURE:")
print(f"   Each notebook contains:")
print(f"   1. Analysis-specific setup (atlas type + polarity)")
print(f"   2. Load cached workflow state")
print(f"   3. Interactive curation GUI launch")
print(f"   4. Progress tracking and auto-save")
print(f"   5. Export curation results")

print(f"\n🎯 BENEFIT: Analysts can work on specific analyses independently")
print(f"   - Focused curation per analysis type")
print(f"   - Parallel processing by multiple analysts")
print(f"   - Clear separation of results")

In [ ]:
# Show example content of an individual analysis notebook
print("📖 EXAMPLE: Individual Notebook Content")
print("=" * 50)

example_notebook_content = '''
# Manual Curation: ISTD POSITIVE
## Project: Demo_Project_2024

### Analysis Summary
- Atlas type: **ISTD**
- Polarity: **POSITIVE**
- Total compounds: **45**
- With MS/MS data: **23**

# Setup and Configuration
ATLAS_TYPE = 'istd'
POLARITY = 'pos' 
ANALYSIS_KEY = 'istd_pos'

# Load workflow state (from cache)
workflow.run_complete_workflow(
    stop_at_stage=wfo.WorkflowStage.PUTATIVE_IDENTIFICATION,
    analysis_subset=[(ATLAS_TYPE, POLARITY)]
)

# Launch curation GUI for this specific analysis
putative_ids = workflow.results.putative_ids[ANALYSIS_KEY]
gui_result = tgui.create_curation_interface(
    plot_data, 
    config,
    auto_save_callback=save_progress
)

# Export results for this analysis
output_file = f'curation_summary_{ATLAS_TYPE}_{POLARITY}.csv'
'''

print("Example notebook structure:")
for line in example_notebook_content.strip().split('\n'):
    if line.strip():
        if line.startswith('#'):
            print(f"🔹 {line}")
        else:
            print(f"   {line}")

print(f"\n🎯 KEY ADVANTAGES:")
print(f"   ✅ Analysis-specific focus (no distractions)")
print(f"   ✅ Independent progress tracking")
print(f"   ✅ Parallel analyst workflows")
print(f"   ✅ Targeted result exports")

## Section 6: Create Workflow Summary Report

Generate a comprehensive summary showing RT correction statistics, putative identification counts by analysis type, and paths to the individual analysis notebooks for manual curation.

In [ ]:
# Generate comprehensive workflow summary
print("📊 COMPREHENSIVE WORKFLOW SUMMARY")
print("=" * 60)

# Get workflow status
workflow_status = workflow.get_workflow_status()
completion_status = workflow.results.get_workflow_completion_status()

print(f"🚀 WORKFLOW STATUS")
print(f"   Current stage: {workflow_status['current_stage']}")
print(f"   Completed stages: {workflow_status['stages_completed']}")
print(f"   Next stage: {workflow_status.get('next_stage', 'Complete')}")

# RT Correction Summary
print(f"\n⚡ RT CORRECTION SUMMARY")
if workflow.results.has_rt_correction_results():
    rt_summary = workflow.results.get_rt_correction_summary()
    print(f"   ✅ Status: Complete")
    print(f"   Models created: {rt_summary['rt_models_count']}")
    print(f"   Methods: {rt_summary['methods_corrected']}")
    print(f"   Corrected atlases: {len([atlas for atlases in rt_summary['corrected_atlases'].values() for chrom in atlases.values() for atlas in chrom.values()])}")
else:
    print(f"   ⏳ Status: Pending")

# Putative Identification Summary  
print(f"\n🔍 PUTATIVE IDENTIFICATION SUMMARY")
if workflow.results.has_putative_identification_results():
    id_summary = workflow.results.get_putative_identification_summary()
    print(f"   ✅ Status: Complete")
    print(f"   Total identifications: {id_summary['total_putative_ids']}")
    print(f"   By analysis type:")
    for atlas_type, count in id_summary['by_atlas_type'].items():
        print(f"      {atlas_type.upper()}: {count}")
    print(f"   With MS/MS data: {id_summary['with_ms2_data']}")
    print(f"   With reference hits: {id_summary['with_reference_hits']}")
else:
    print(f"   ⏳ Status: Pending")

# Individual Notebooks Summary
print(f"\n📓 INDIVIDUAL NOTEBOOKS")
expected_notebooks = []
for analysis_atlas in workflow.atlas_data.get('analysis_atlases', []):
    atlas_type = analysis_atlas['atlas_type']
    polarity = analysis_atlas['polarity'] 
    notebook_name = f"curation_{atlas_type}_{polarity}.ipynb"
    expected_notebooks.append((atlas_type, polarity, notebook_name))

print(f"   Expected notebooks: {len(expected_notebooks)}")
for atlas_type, polarity, notebook_name in expected_notebooks:
    print(f"      📝 {notebook_name} ({atlas_type.upper()} {polarity.upper()})")

# Cache Status
print(f"\n💾 CACHE STATUS")
cache_exists = {
    'RT Correction': workflow.cache_manager.has_rt_correction_cache(),
    'Putative IDs': workflow.cache_manager.has_putative_identifications_cache(), 
    'Manual Curation': workflow.cache_manager.has_manual_curation_cache()
}

for cache_type, exists in cache_exists.items():
    status = "✅ Available" if exists else "❌ Not cached"
    print(f"   {cache_type}: {status}")

print(f"\n" + "=" * 60)
print(f"🎯 WORKFLOW READY FOR MANUAL CURATION")
print(f"   Next step: Run individual analysis notebooks for manual curation")
print(f"   Each analyst can work independently on their assigned analyses")

## Workflow Comparison: Old vs New

### Old Iterative Workflow
```python
# Old way: Multiple RT models, complex iteration
for atlas_type in ['qc', 'istd', 'ema']:
    for chrom in ['hilic']:
        for polarity in ['positive', 'negative']:
            # Separate RT correction for each combination
            rt_model = create_rt_model(atlas_type, chrom, polarity)
            corrected_atlas = apply_rt_correction(rt_model)
            putative_ids = run_analysis(corrected_atlas)
```

### New Single-Model Workflow  
```python
# New way: One RT model, multiple atlas application
rt_model = create_single_rt_model(rt_align_atlas)  # One model
for analysis_atlas in all_analysis_atlases:       # Apply to all
    corrected_atlas = apply_rt_correction(rt_model, analysis_atlas)
    putative_ids = run_analysis(corrected_atlas)
    create_individual_notebook(analysis_atlas)    # Individual notebooks
```

### Key Improvements
1. **Single RT Model**: One model applied consistently across all analyses
2. **Analysis Subset Support**: `analysis_subset=[('ISTD', 'NEG'), ('ISTD', 'POS')]`
3. **Individual Notebooks**: Separate curation notebooks per analysis type/polarity  
4. **Parallel Workflows**: Multiple analysts can work simultaneously
5. **Focused Curation**: Each notebook contains only relevant compounds

In [ ]:
# Usage Examples for the Updated Workflow
print("💡 USAGE EXAMPLES")
print("=" * 50)

print("📝 Example 1: Run Complete Workflow (All Analyses)")
example1 = '''
# Run all computational stages, create individual notebooks
workflow.run_complete_workflow(
    stop_at_stage=wfo.WorkflowStage.PUTATIVE_IDENTIFICATION,
    create_individual_notebooks=True
)
'''
print(example1)

print("📝 Example 2: Run Subset of Analyses")  
example2 = '''
# Focus on ISTD analyses only
workflow.run_complete_workflow(
    stop_at_stage=wfo.WorkflowStage.PUTATIVE_IDENTIFICATION,
    analysis_subset=[('istd', 'pos'), ('istd', 'neg')],
    create_individual_notebooks=True
)
'''
print(example2)

print("📝 Example 3: Debug Single Analysis")
example3 = '''
# Test single analysis for debugging
workflow.run_complete_workflow(
    stop_at_stage=wfo.WorkflowStage.PUTATIVE_IDENTIFICATION,
    analysis_subset=[('qc', 'pos')],
    create_individual_notebooks=True
)
'''
print(example3)

print("📝 Example 4: Use Cached Results")
example4 = '''
# Quick re-run using cached RT correction
workflow.config['WORKFLOWS']['HILICZ']['RT_ALIGN']['PARAMS']['use_rt_correction_cache'] = True
workflow.run_complete_workflow(
    stop_at_stage=wfo.WorkflowStage.PUTATIVE_IDENTIFICATION,
    create_individual_notebooks=True
)
'''
print(example4)

print("\n🎯 FINAL SUMMARY")
print("=" * 50)
print("✅ Updated workflow successfully demonstrates:")
print("   • Single RT alignment model creation")
print("   • Application to multiple analysis atlases") 
print("   • Analysis subset support for focused work")
print("   • Individual notebook generation per analysis")
print("   • Improved analyst workflow efficiency")
print("\n🚀 Ready for production use with real data!")